In [72]:
from meshing import Mesh2D
import matplotlib.pyplot as plt
from problems import continuous_coefficient_2d
from discretization import discretize
import numpy as np
import scipy.sparse as sps
import pandas as pd
import tqdm

In [73]:
class CG:
    def __init__(
        self,
        preconditioner=None,
        rtol: float = 1e-7,
        atol: float = 0,
        maxiter: int | None = 1000,
        estimate_cond: bool = False,
    ):
        self.preconditioner = preconditioner
        self.rtol = rtol
        self.atol = atol
        self.maxiter = maxiter
        self.estimate_cond = estimate_cond

    def __str__(self):
        return f"CG({self.preconditioner or ''}, bsr_matmul={self.bsr_matmul})"

    def setup(self, matrix, *args, **kwargs) -> None:
        if self.preconditioner is not None:
            self.preconditioner.setup(matrix, *args, **kwargs)

        self.matrix = matrix

    def solve(self, rhs, x0=None):
        atol = max(self.atol, self.rtol * np.linalg.norm(rhs).item())
        n = len(rhs)
        maxiter = self.maxiter or n * 10
        x = x0 if x0 is not None else np.zeros_like(rhs)
        if self.preconditioner is not None and hasattr(self.preconditioner, "T0"):
            x += self.preconditioner.T0(rhs)
        r = rhs - self.matrix @ x
        rho_prev, p = None, None

        residual_norms = []
        preconditioner_metadata = []
        if self.estimate_cond:
            alphas = []
            betas = []
        for i in range(maxiter):
            res_norm = np.linalg.norm(r).item()
            residual_norms.append(res_norm)
            if res_norm < atol:
                break

            (z, metadata) = (
                self.preconditioner.solve(r)
                if self.preconditioner is not None
                else (r, {})
            )

            preconditioner_metadata.append(metadata)
            rho_cur = np.dot(r, z)
            if i > 0:
                beta = rho_cur / rho_prev
                p *= beta
                p += z
            else:
                p = np.empty_like(r)
                p[:] = z[:]

            q = self.matrix @ p
            alpha = rho_cur / np.dot(p, q)
            x += alpha * p
            r -= alpha * q
            rho_prev = rho_cur

            if self.estimate_cond:
                if i > 0:
                    betas.append(beta)
                alphas.append(alpha)

        metadata = {
            "iterations": len(residual_norms) - 1,
            "residual norms": residual_norms,
            "preconditioner metadata": preconditioner_metadata,
        }

        if self.estimate_cond:
            lmat = np.zeros(
                (len(alphas), len(alphas)),
                dtype=rhs.dtype,
            )
            for i in range(len(alphas)):
                lmat[i, i] = 1 / alphas[i]
                if i > 0:
                    lmat[i, i] += betas[i - 1] / alphas[i - 1]
                    lmat[i - 1, i] = lmat[i, i - 1] = (
                        betas[i - 1] ** (1 / 2) / alphas[i - 1]
                    )
            metadata["condition number estimate"] = np.linalg.cond(lmat, p=2).item()

        if i + 1 == maxiter:
            raise Exception(metadata)

        return x, metadata

    def destroy(self) -> None:
        if self.preconditioner is not None:
            self.preconditioner.destroy()
            self.preconditioner = None
        self.matrix = None

In [74]:
class OverlappingASM:
    def setup(
        self,
        matrix,
        fine_to_solvers: list[list[int]],
        fine_to_coarse: np.ndarray,
        solvers_to_fine: list[list[int]],
        coarse_to_fine: np.ndarray,
        dofs_per_element: int = 3,
    ):
        self.fine_to_solvers = fine_to_solvers
        self.solvers_to_fine = solvers_to_fine
        self.coarse_to_fine = coarse_to_fine
        self.dofs_per_element = dofs_per_element

        self.n_solvers = len(solvers_to_fine)

        self.solver_dofs = [[] for _ in range(self.n_solvers)]
        self.local_solvers = [None] * self.n_solvers
        for i in range(self.n_solvers):
            for el in solvers_to_fine[i]:
                for d in range(dofs_per_element):
                    self.solver_dofs[i].append(el * dofs_per_element + d)
            solver_matrix = matrix[self.solver_dofs[i]][:, self.solver_dofs[i]]
            self.local_solvers[i] = sps.linalg.splu(solver_matrix.tocsc())

        matrix_coo = matrix.tocoo()
        coarse_row = fine_to_coarse[matrix_coo.row // dofs_per_element]
        coarse_col = fine_to_coarse[matrix_coo.col // dofs_per_element]
        coarse_data = matrix_coo.data
        coarse_matrix = sps.coo_matrix(
            (coarse_data, (coarse_row, coarse_col)),
            shape=(len(coarse_to_fine), len(coarse_to_fine)),
        ).tocsc()
        self.coarse_solver = sps.linalg.splu(coarse_matrix)

    def destroy(self):
        pass

    def solve(self, x: np.ndarray):
        x_coarse = (
            x.reshape(-1, self.dofs_per_element)
            .sum(axis=1)[self.coarse_to_fine]
            .sum(axis=1)
        )
        y_coarse = self.coarse_solver.solve(x_coarse)
        y_hlp = np.zeros(len(x) // self.dofs_per_element)
        y_hlp[self.coarse_to_fine] = y_coarse[:, None]
        y = y_hlp.repeat(self.dofs_per_element, axis=0)
        for i, local_solver in enumerate(self.local_solvers):
            local_x = x[self.solver_dofs[i]]
            local_y = local_solver.solve(local_x)
            y[self.solver_dofs[i]] += local_y
        return y, {}

In [75]:
def fine_to_mesh(fine_mesh, n: int, N: int, overlap_n: int):
    v = fine_mesh.vertices[fine_mesh.elements]
    coords = (v.mean(axis=1) * n).astype(int)

    def coords_to_mesh(x, y):
        base_solver = (x // (n // N)) * N + (y // (n // N))
        in_solver_coords = (x % (n // N), y % (n // N))
        solvers = [base_solver]
        if in_solver_coords[1] < overlap_n and base_solver % N != 0:
            solvers.append(base_solver - 1)
        if in_solver_coords[1] >= (n // N - overlap_n) and base_solver % N != N - 1:
            solvers.append(base_solver + 1)
        if in_solver_coords[0] < overlap_n and base_solver >= N:
            solvers.append(base_solver - N)
        if in_solver_coords[0] >= (n // N - overlap_n) and base_solver < N * (N - 1):
            solvers.append(base_solver + N)
        # now we have to check the corners
        if (
            in_solver_coords[0] < overlap_n
            and in_solver_coords[1] < overlap_n
            and base_solver >= N
            and base_solver % N != 0
        ):
            solvers.append(base_solver - N - 1)
        if (
            in_solver_coords[0] < overlap_n
            and in_solver_coords[1] >= (n // N - overlap_n)
            and base_solver >= N
            and base_solver % N != N - 1
        ):
            solvers.append(base_solver - N + 1)
        if (
            in_solver_coords[0] >= (n // N - overlap_n)
            and in_solver_coords[1] < overlap_n
            and base_solver < N * (N - 1)
            and base_solver % N != 0
        ):
            solvers.append(base_solver + N - 1)
        if (
            in_solver_coords[0] >= (n // N - overlap_n)
            and in_solver_coords[1] >= (n // N - overlap_n)
            and base_solver < N * (N - 1)
            and base_solver % N != N - 1
        ):
            solvers.append(base_solver + N + 1)
        return solvers

    return [coords_to_mesh(x, y) for x, y in coords]

In [76]:
def test(n, N, NN, overlap_n):
    assert n % N == 0
    assert N % NN == 0
    assert overlap_n >= 0 and overlap_n <= n // N // 2

    fine_mesh = Mesh2D.unit_square_uniform(n, n)

    fine_to_solvers = fine_to_mesh(fine_mesh, n, N, overlap_n)
    fine_to_coarse = np.array(fine_to_mesh(fine_mesh, n, NN, 0)).flatten()

    solvers_to_fine = [[] for _ in range(N * N)]
    for i, solvers in enumerate(fine_to_solvers):
        for solver in solvers:
            solvers_to_fine[solver].append(i)

    coarse_to_fine = [[] for _ in range(NN * NN)]
    for i, coarse in enumerate(fine_to_coarse):
        coarse_to_fine[coarse].append(i)
    coarse_to_fine = np.array(coarse_to_fine)

    discrete_problem = discretize(
        problem=continuous_coefficient_2d.problem,
        mesh=fine_mesh,
    )

    preconditioner = OverlappingASM()
    cg_solver = CG(
        preconditioner=preconditioner, estimate_cond=True, rtol=1e-6, maxiter=2000
    )
    cg_solver.setup(
        matrix=discrete_problem.exact_form_matrix,
        fine_to_solvers=fine_to_solvers,
        fine_to_coarse=fine_to_coarse,
        solvers_to_fine=solvers_to_fine,
        coarse_to_fine=coarse_to_fine,
        dofs_per_element=3,
    )

    test_metadata = {
        "n": n,
        "N": N,
        "NN": NN,
        "overlap_n": overlap_n,
    }

    try:
        rhs = np.random.rand(discrete_problem.load_vector.shape[0])
        rhs = (rhs - 0.5) * 2

        sol, metadata = cg_solver.solve(rhs)
        residual_norm = np.linalg.norm(discrete_problem.exact_form_matrix @ sol - rhs)
        return {
            **test_metadata,
            "residual norm": residual_norm,
            "iterations": metadata["iterations"],
            "condition number": metadata.get("condition number estimate", None),
        }
    except Exception as e:
        return {
            **test_metadata,
            "exception": str(e),
        }

In [77]:
test_cases = []
for m in range(4, 7):
    for M in range(4, m + 1):
        for MM in range(4, M + 1):
            n = 2**m
            N = 2**M
            NN = 2**MM
            test_cases.append((n, N, NN, 0))
            for overlap_m in range(m - M):
                test_cases.append((n, N, NN, 2**overlap_m))

len(test_cases)

15

In [78]:
results = []
for test_case in tqdm.tqdm(test_cases):
    results.append(test(*test_case))

100%|██████████| 15/15 [00:21<00:00,  1.40s/it]


In [79]:
df = pd.DataFrame(results)
df.to_csv("overlapping_asm_results.csv", index=False)
df

,n,N,NN,overlap_n,residual norm,iterations,condition number
0,16,16,16,0,0.000015,32,20.353965
1,32,16,16,0,0.000043,41,36.012252
2,32,16,16,1,0.000041,35,35.839584
3,32,32,16,0,0.000044,41,34.771188
4,32,32,32,0,0.000040,33,21.636381
5,64,16,16,0,0.000088,57,71.912541
6,64,16,16,1,0.000088,49,72.336906
7,64,16,16,2,0.000052,44,61.296318
8,64,32,16,0,0.000074,58,72.490535
9,64,32,16,1,0.000090,57,86.208107


In [80]:
def get_order(row):
    h = 1 / row["n"]
    H = 1 / row["N"]
    HH = 1 / row["NN"]
    delta = row["overlap_n"] / row["n"]
    return HH * HH / H / max(h, delta)


df["estimate"] = df.apply(get_order, axis=1)
df["ratio"] = df["condition number"] / df["estimate"]
df[(df.N == df.NN)]

,n,N,NN,overlap_n,residual norm,iterations,condition number,estimate,ratio
0,16,16,16,0,0.000015,32,20.353965,1.0,20.353965
1,32,16,16,0,0.000043,41,36.012252,2.0,18.006126
2,32,16,16,1,0.000041,35,35.839584,2.0,17.919792
4,32,32,32,0,0.000040,33,21.636381,1.0,21.636381
5,64,16,16,0,0.000088,57,71.912541,4.0,17.978135
6,64,16,16,1,0.000088,49,72.336906,4.0,18.084226
7,64,16,16,2,0.000052,44,61.296318,2.0,30.648159
10,64,32,32,0,0.000077,42,36.552260,2.0,18.276130
11,64,32,32,1,0.000062,45,45.676940,2.0,22.838470
14,64,64,64,0,0.000090,33,22.086110,1.0,22.086110
